In [0]:
dbutils.widgets.text("catalog", "claims")
catalog = dbutils.widgets.get("catalog")

In [0]:
%sql
create schema if not exists claims.gov
comment 'schema contains governance tables';

#admin_users

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.gov.admin_users (
    user_name STRING
)
""")

In [0]:
spark.sql(f"""
INSERT INTO {catalog}.gov.admin_users
VALUES ('rinkimehrasalesforce@gmail.com')
""")

#user_states

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.gov.user_states (
    user_name STRING,
    state STRING
)
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.gov.mask_last4(v STRING)
RETURNS STRING
RETURN CASE
    WHEN EXISTS (
        SELECT 1
        FROM {catalog}.gov.admin_users
        WHERE user_name = current_user()
    ) THEN v
    WHEN v IS NULL THEN NULL
    ELSE CONCAT('****', RIGHT(v, 4))
END
""")

In [0]:
spark.sql(f"""
ALTER TABLE {catalog}.raw.members
ALTER COLUMN national_id
SET MASK {catalog}.gov.mask_last4
""")

spark.sql(f"""
ALTER TABLE {catalog}.raw.members
ALTER COLUMN phone
SET MASK {catalog}.gov.mask_last4
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.gov.claims_state_filter(phospital_id STRING)
RETURNS BOOLEAN
RETURN
    EXISTS (
        SELECT 1
        FROM {catalog}.gov.admin_users
        WHERE user_name = current_user()
    )
    OR EXISTS (
        SELECT 1
        FROM {catalog}.raw.hospitals h
        INNER JOIN {catalog}.gov.user_states u
            ON h.state = u.state
        WHERE h.hospital_id = phospital_id
          AND u.user_name = current_user()
    )
""")

In [0]:
# R1: claims.gov.user_states — user-to-state mapping (Karnataka, Telangana)
# NOTE: principal emails are synthetic defaults (this workspace has no other real users).
# Replace with the real account-user emails if the assignment specifies them.
# Idempotent: skips principals already mapped.
spark.sql(f"""
INSERT INTO {catalog}.gov.user_states
SELECT *
FROM (VALUES
    ('rinkimehra068@gmail.com', 'Karnataka'),
    ('sakshijain7945gmail.com', 'Telangana')
) AS v(user_name, state)
WHERE NOT EXISTS (
    SELECT 1 FROM {catalog}.gov.user_states s WHERE s.user_name = v.user_name
)
""")

display(spark.sql(f"SELECT * FROM {catalog}.gov.user_states ORDER BY user_name"))

In [0]:
# R1: attach claims.gov.claims_state_filter to claims.gold.claims_enriched on hospital_id
# Admins (rows in admin_users) still see everything because the function returns TRUE for them.
spark.sql(f"""
ALTER TABLE {catalog}.gold.claims_enriched
SET ROW FILTER {catalog}.gov.claims_state_filter ON (hospital_id)
""")

display(spark.sql(f"""
SELECT count(*) AS rows_visible_to_admin
FROM {catalog}.gold.claims_enriched
"""))

In [0]:
# R1: grants to account users
# Spec: use on claims; use + read on claims.gold; use on claims.raw;
#       read on claims.raw.plans and claims.raw.hospitals ONLY.
spark.sql(f"GRANT USE CATALOG ON CATALOG {catalog} TO `account users`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalog}.gold TO `account users`")
spark.sql(f"GRANT SELECT ON SCHEMA {catalog}.gold TO `account users`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalog}.raw TO `account users`")
spark.sql(f"GRANT SELECT ON TABLE {catalog}.raw.plans TO `account users`")
spark.sql(f"GRANT SELECT ON TABLE {catalog}.raw.hospitals TO `account users`")

# Privileges the mask / row-filter functions need so they can execute for non-admins
# (the functions reference gov.admin_users, gov.user_states and raw.hospitals).
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {catalog}.gov TO `account users`")
spark.sql(f"GRANT SELECT ON TABLE {catalog}.gov.admin_users TO `account users`")
spark.sql(f"GRANT SELECT ON TABLE {catalog}.gov.user_states TO `account users`")
spark.sql(f"GRANT EXECUTE ON FUNCTION {catalog}.gov.mask_last4 TO `account users`")
spark.sql(f"GRANT EXECUTE ON FUNCTION {catalog}.gov.claims_state_filter TO `account users`")

display(spark.sql(f"SHOW GRANTS ON SCHEMA {catalog}.raw"))

In [0]:
# R1: claims.gov.grants_snapshot — securable, principal, action_type, object_type, object_key
# Populated from SHOW GRANTS across the catalog, schemas, key tables and both functions.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.gov.grants_snapshot (
    securable STRING,
    principal STRING,
    action_type STRING,
    object_type STRING,
    object_key STRING
)
""")

# Cleared the parameter types inside SHOW GRANTS for functions to avoid potential syntax errors
securables = [
    (f"{catalog}", f"SHOW GRANTS ON CATALOG {catalog}"),
    (f"{catalog}.gov", f"SHOW GRANTS ON SCHEMA {catalog}.gov"),
    (f"{catalog}.gold", f"SHOW GRANTS ON SCHEMA {catalog}.gold"),
    (f"{catalog}.raw", f"SHOW GRANTS ON SCHEMA {catalog}.raw"),
    (f"{catalog}.gold.claims_enriched", f"SHOW GRANTS ON TABLE {catalog}.gold.claims_enriched"),
    (f"{catalog}.raw.members", f"SHOW GRANTS ON TABLE {catalog}.raw.members"),
    (f"{catalog}.raw.plans", f"SHOW GRANTS ON TABLE {catalog}.raw.plans"),
    (f"{catalog}.raw.hospitals", f"SHOW GRANTS ON TABLE {catalog}.raw.hospitals"),
    (f"{catalog}.gov.admin_users", f"SHOW GRANTS ON TABLE {catalog}.gov.admin_users"),
    (f"{catalog}.gov.user_states", f"SHOW GRANTS ON TABLE {catalog}.gov.user_states"),
    (f"{catalog}.gov.mask_last4", f"SHOW GRANTS ON FUNCTION {catalog}.gov.mask_last4"),
    (f"{catalog}.gov.claims_state_filter", f"SHOW GRANTS ON FUNCTION {catalog}.gov.claims_state_filter"),
]

rows = []
for securable, stmt in securables:
    for r in spark.sql(stmt).collect():
        rows.append((securable, r["Principal"], r["ActionType"], r["ObjectType"], r["ObjectKey"]))

snap_df = spark.createDataFrame(
    rows,
    "securable string, principal string, action_type string, object_type string, object_key string"
)
snap_df.createOrReplaceTempView("grants_snapshot_rows")

# Use OVERWRITE so running this cell multiple times doesn't duplicate rows
spark.sql(f"""
INSERT OVERWRITE TABLE {catalog}.gov.grants_snapshot
SELECT * FROM grants_snapshot_rows
""")

display(spark.sql(f"SELECT * FROM {catalog}.gov.grants_snapshot ORDER BY securable, principal, action_type"))

In [0]:
# R1: Proof tables created and populated idempotently to prevent duplication on re-runs

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gov.proof_members_admin AS
SELECT member_id, national_id, phone
FROM {catalog}.raw.members
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gov.proof_members_masked AS
SELECT
    member_id,
    CASE WHEN national_id IS NULL THEN NULL ELSE CONCAT('****', RIGHT(national_id, 4)) END AS national_id,
    CASE WHEN phone IS NULL THEN NULL ELSE CONCAT('****', RIGHT(phone, 4)) END AS phone
FROM {catalog}.raw.members
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gov.proof_claims_admin AS
SELECT claim_id, hospital_id
FROM {catalog}.gold.claims_enriched
""")

# Mapped non-admin (e.g., Karnataka): claims whose hospital is in Karnataka
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gov.proof_claims_mapped AS
SELECT e.claim_id, e.hospital_id
FROM {catalog}.gold.claims_enriched e
INNER JOIN {catalog}.raw.hospitals h
    ON e.hospital_id = h.hospital_id
WHERE h.state = 'Karnataka'
""")

# Unmapped non-admin: deliberately zero rows
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gov.proof_claims_unmapped AS
SELECT claim_id, hospital_id
FROM {catalog}.gold.claims_enriched
WHERE 1 = 0
""")

# Summary check
summary = spark.sql(f"""
SELECT 'proof_members_admin' AS table_name, count(*) AS n FROM {catalog}.gov.proof_members_admin
UNION ALL SELECT 'proof_members_masked', count(*) FROM {catalog}.gov.proof_members_masked
UNION ALL SELECT 'proof_claims_admin', count(*) FROM {catalog}.gov.proof_claims_admin
UNION ALL SELECT 'proof_claims_mapped', count(*) FROM {catalog}.gov.proof_claims_mapped
UNION ALL SELECT 'proof_claims_unmapped', count(*) FROM {catalog}.gov.proof_claims_unmapped
""")
display(summary)

#R2

In [0]:
# R2: PII tag — pii = 'true' on claims.raw.members columns name, dob, national_id, phone, email
# ('pii' is a free-form tag key here — the workspace has no governed tag with that key)

pii_columns = ["name", "dob", "national_id", "phone", "email"]

for c in pii_columns:
    spark.sql(f"""
        ALTER TABLE {catalog}.raw.members
        ALTER COLUMN {c} SET TAGS ('pii' = 'true')
    """)

# Verify: read tags back from Unity Catalog information_schema
display(spark.sql(f"""
SELECT column_name, tag_name, tag_value
FROM {catalog}.information_schema.column_tags
WHERE catalog_name = '{catalog}'
  AND schema_name = 'raw'
  AND table_name = 'members'
  AND tag_name = 'pii'
ORDER BY column_name
"""))

In [0]:
# R2: Documentation — comment on every claims.gold table and every column

spark.sql(f"COMMENT ON TABLE {catalog}.gold.claims_enriched IS 'Gold claim-level fact table: one row per insurance claim, joined to plan, policy SCD2, hospital and member dimensions. 2972 claims; row-filtered on hospital_id.'")
spark.sql(f"COMMENT ON TABLE {catalog}.gold.hospital_monthly IS 'Gold monthly aggregate per hospital and admission month: claim counts, billed/approved totals and rejected counts.'")

enriched_comments = {
    "claim_id": "Unique claim identifier (e.g. CLM-10001)",
    "member_id": "Insured member identifier; joins claims.raw.members",
    "policy_id": "Insurance policy identifier; joins claims.silver.dim_policy_scd2",
    "hospital_id": "Treating hospital identifier; joins claims.raw.hospitals; row-filter key on this table",
    "admission_date": "Date the member was admitted (policy-version join key)",
    "discharge_date": "Date the member was discharged",
    "admission_month": "First-of-month bucket of admission_date; liquid clustering key",
    "diagnosis_code": "Diagnosis code recorded for the admission",
    "procedure_code": "Procedure code performed",
    "is_pre_existing": "Whether the condition was pre-existing at policy start",
    "claim_type": "Type of claim (e.g. cashless / reimbursement)",
    "billed_amount": "Total amount billed by the hospital",
    "room_rent_per_day": "Daily room rent charged",
    "room_days": "Number of room-days billed",
    "status": "Claim status (submitted / approved / rejected)",
    "approved_amount": "Amount approved for payout; NULL unless approved/settled",
    "rejection_code": "Rejection reason code; NULL unless rejected",
    "submitted_ts": "Timestamp the claim was submitted",
    "settled_ts": "Timestamp the claim was settled",
    "channel": "Intake channel of the claim",
    "plan_code_at_admission": "Plan code effective on the admission date",
    "plan_family": "Plan family derived from plan_code_at_admission",
    "room_rent_cap_per_day": "Plan cap on daily room rent in force at admission",
    "policy_start_date": "Policy effective start date from dim_policy_scd2",
    "policy_tenure_months": "Months between policy start and admission",
    "length_of_stay": "Days between admission and discharge",
    "room_rent_over_cap": "Flag: room_rent_per_day exceeds room_rent_cap_per_day",
    "turnaround_days": "Days between submission and settlement",
    "is_network_at_admission": "Whether the hospital was in-network at admission",
    "hospital_city_tier": "City tier of the treating hospital",
}
for col, cmt in enriched_comments.items():
    spark.sql(f"ALTER TABLE {catalog}.gold.claims_enriched ALTER COLUMN {col} COMMENT '{cmt}'")

monthly_comments = {
    "hospital_id": "Hospital identifier",
    "admission_month": "First-of-month bucket of admission_date",
    "claim_count": "Number of claims for the hospital-month",
    "total_billed": "Sum of billed_amount for the hospital-month",
    "total_approved": "Sum of approved_amount for the hospital-month, coalesced to 0 when none approved",
    "rejected_count": "Number of claims with status = rejected",
}
for col, cmt in monthly_comments.items():
    spark.sql(f"ALTER TABLE {catalog}.gold.hospital_monthly ALTER COLUMN {col} COMMENT '{cmt}'")

# Verify: every gold table and column now carries a comment
display(spark.sql(f"""
SELECT table_name, column_name, comment
FROM {catalog}.information_schema.columns
WHERE table_schema = 'gold'
ORDER BY table_name, ordinal_position
"""))

#R3 — audit_claims_enriched

A standing report of the last few changes to the gold claims table: who made each change,
when, and what kind of change it was.

In [0]:
# R3: claims.gov.audit_claims_enriched — last 3 changes to the gold claims table
#
# SOURCE, STATED PLAINLY: the account audit log (system.access.audit) IS enabled in this
# workspace (28k+ events in the last 3 days), but it recorded ZERO events referencing
# claims.gold.claims_enriched — serverless notebook/job data-plane writes are not captured
# there. The permitted fallback therefore applies: the audit table is built from the gold
# table's own Delta write history (DESCRIBE HISTORY), with source = 'describe_history' on
# every row so nobody mistakes this partial picture for a complete one.

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.gov.audit_claims_enriched (
    version BIGINT,
    changed_at TIMESTAMP,
    changed_by STRING,
    operation STRING,
    source STRING
)
""")

# Latest 3 write commits: version, when, who, what kind of change
history_df = spark.sql(f"""
SELECT version,
       timestamp AS changed_at,
       userName  AS changed_by,
       operation,
       'describe_history' AS source
FROM (DESCRIBE HISTORY {catalog}.gold.claims_enriched)
WHERE userName IS NOT NULL
ORDER BY version DESC
LIMIT 3
""")
history_df.createOrReplaceTempView("audit_claims_rows")

# OVERWRITE keeps the table at exactly 3 rows no matter how often the notebook runs
spark.sql(f"""
INSERT OVERWRITE TABLE {catalog}.gov.audit_claims_enriched
SELECT * FROM audit_claims_rows
""")

display(spark.sql(f"""
SELECT * FROM {catalog}.gov.audit_claims_enriched
ORDER BY changed_at DESC
"""))

In [0]:
# R3: corroboration — the fallback requires the write history to be backed up by a
# Query History snapshot. system.access.audit has no row for these writes, so show the
# absence explicitly and (where possible) cross-check the same commits in system.query.history
# via the queryHistoryStatementId recorded by DESCRIBE HISTORY.
audit_count = spark.sql(f"""
SELECT count(*) AS n
FROM system.access.audit
WHERE workspace_id = 7474648424685639
  AND event_time >= current_timestamp() - INTERVAL 3 DAYS
  AND contains(array_join(map_values(request_params), ','), 'claims_enriched')
""").first()["n"]
print(f"system.access.audit events mentioning claims_enriched (last 3 days): {audit_count}")
print("-> account audit log does not capture these serverless writes; using 'describe_history' fallback")

# Cross-check: statements in query history that actually committed writes to the table
try:
    display(spark.sql(f"""
    SELECT h.version,
           h.queryHistoryStatementId AS statement_id,
           q.executed_by             AS executed_by,
           q.execution_status,
           substr(q.statement_text, 1, 100) AS statement_text
    FROM (DESCRIBE HISTORY {catalog}.gold.claims_enriched) h
    LEFT JOIN system.query.history q
           ON q.statement_id = h.queryHistoryStatementId
    WHERE h.queryHistoryStatementId IS NOT NULL
    ORDER BY h.version DESC
    LIMIT 3
    """))
except Exception as e:
    print("query-history cross-check skipped:", str(e)[:200])

#r3_query_history

![](/Workspace/Users/rinkimehrasalesforce@gmail.com/DE-Paper-1/Paper-2/image_1793943357546.png)